# MP Data Preparation (Lucas Edit)

## Notebook Contract
- Primary input files:
  - `./Datasets/Raw Dataset/OriginalDataMP.csv`
  - `./Datasets/Raw Dataset/yaws_physical_properties_inhouse.csv` (in-house digitization; authoritative Yaw's property values, with locally recovered SMILES)
- Adapter behavior:
  - Keep the organic section of the in-house file
  - Use its `SMILES`, `freezing_point_K`, and `freezing_point_code` fields directly
  - Restore the previously user-verified physical-book row for 4-methoxypteridine (record 11441, 468.15 K), which is absent from the cleaned master because of the earlier structural exception
  - Rename `freezing_point_K` to `FP_K` and retain only `SMILES`, `FP_K`, and `FP_code` before the existing pipeline
- Core behaviors and outputs remain unchanged.


## 1. Setup and Load Raw Data


In [ ]:
import os
import pandas as pd
from collections import Counter
from rdkit import Chem
from sklearn.model_selection import train_test_split

# Configuration
ORIGINAL_RAW_PATH = './Datasets/Raw Dataset/OriginalDataMP.csv'
YAWS_RAW_PATH = './Datasets/Raw Dataset/yaws_physical_properties_inhouse.csv'
CURATED_DIR = './Datasets/Curated Dataset(s)'
TRAINABLE_DIR = './Datasets/trainable_data'
CURATED_FILENAME = 'CleanedDataMP.csv'
TRAINABLE_FILENAME = 'MP_chemprop_train_file.csv'
RANDOM_SEED = 42
MAX_HEAVY_ATOMS = 40

original_raw_df = pd.read_csv(ORIGINAL_RAW_PATH)
yaws_source_df = pd.read_csv(YAWS_RAW_PATH, dtype=str, keep_default_na=False)

required_original_cols = {'original SMILES', 'MP'}
required_yaws_source_cols = {
    'section', 'record_no', 'SMILES', 'freezing_point_K', 'freezing_point_code'
}
missing_original = required_original_cols - set(original_raw_df.columns)
missing_yaws_source = required_yaws_source_cols - set(yaws_source_df.columns)
if missing_original:
    raise ValueError(f'Missing required columns in OriginalDataMP.csv: {sorted(missing_original)}')
if missing_yaws_source:
    raise ValueError(f"Missing required columns in in-house Yaw's file: {sorted(missing_yaws_source)}")

yaws_source_core = yaws_source_df.loc[
    yaws_source_df['section'].eq('organic'),
    ['record_no', 'SMILES', 'freezing_point_K', 'freezing_point_code']
].copy()

# Source-specific in-house corrections are omitted from this publication draft.
verified_inhouse_overrides = pd.DataFrame(columns=[
    'record_no', 'SMILES', 'freezing_point_K', 'freezing_point_code'
])
missing_override_ids = verified_inhouse_overrides.loc[
    ~verified_inhouse_overrides['record_no'].isin(yaws_source_core['record_no'])
]
if not missing_override_ids.empty:
    yaws_source_core = pd.concat([yaws_source_core, missing_override_ids], ignore_index=True)

yaws_raw_df = yaws_source_core.rename(columns={
    'freezing_point_K': 'FP_K',
    'freezing_point_code': 'FP_code',
})[['SMILES', 'FP_K', 'FP_code']]
yaws_raw_df = yaws_raw_df.loc[yaws_raw_df['SMILES'].astype(str).str.strip().ne('')].copy()

print(f'Loaded Original rows: {len(original_raw_df)}')
print(f"Loaded in-house Yaw's organic rows: {len(yaws_source_core)}")
print(f"In-house Yaw's rows with SMILES: {len(yaws_raw_df)}")


## 2. Canonicalize SMILES and Build Union Baseline (Yaw's Priority)


In [ ]:
def canonicalize_smiles(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

# Original dataset prep
orig = original_raw_df.copy()
if {'excluded or corrected', 'reason for exclusion'}.issubset(orig.columns):
    orig = orig[orig['excluded or corrected'].isna() & orig['reason for exclusion'].isna()].copy()

orig['MP'] = pd.to_numeric(orig['MP'], errors='coerce')
orig = orig.dropna(subset=['original SMILES', 'MP']).copy()
orig['SMILES'] = orig['original SMILES'].apply(canonicalize_smiles)
orig = orig.dropna(subset=['SMILES']).copy()
orig = orig[['SMILES', 'MP']].copy()
orig['source'] = 'original'

# Yaw's dataset prep (FP_K -> MP in degC for consistent union)
yaws = yaws_raw_df.copy()
yaws['FP_K'] = pd.to_numeric(yaws['FP_K'], errors='coerce')
yaws = yaws.dropna(subset=['SMILES', 'FP_K']).copy()
yaws['SMILES'] = yaws['SMILES'].apply(canonicalize_smiles)
yaws = yaws.dropna(subset=['SMILES']).copy()
yaws['MP'] = yaws['FP_K'] - 273.15
yaws = yaws[['SMILES', 'MP']].copy()
yaws['source'] = 'yaws'

# Union with Yaw's precedence on duplicate canonical SMILES
combined = pd.concat([yaws, orig], ignore_index=True)
combined['source_priority'] = combined['source'].map({'yaws': 0, 'original': 1}).fillna(2)
combined = combined.sort_values(['SMILES', 'source_priority']).drop_duplicates(subset='SMILES', keep='first')
combined = combined[['SMILES', 'MP', 'source']].reset_index(drop=True)

if combined.empty:
    raise ValueError('Combined baseline dataset is empty after union/deduplication.')

print('Baseline union summary:')
print(f'- Original retained pre-union: {len(orig)}')
print(f"- Yaws retained pre-union: {len(yaws)}")
print(f"- Combined retained post-union (Yaw's priority): {len(combined)}")
print(f"- Unique SMILES in combined: {combined['SMILES'].nunique()}")
print(f"- Rows sourced from Yaws after union: {(combined['source'] == 'yaws').sum()}")
print(f"- Rows sourced from Original after union: {(combined['source'] == 'original').sum()}")


## 3. pKa-Style Cleansing Pipeline


In [ ]:
def clean_smiles_dataset(df, output_path, threshold=0.01, max_heavy_atoms=40):
    smiles_col = None
    for candidate in ['SMILES', 'smiles']:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")

    def canonicalize_smiles_inner(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except Exception:
            return None

    def is_organic(mol):
        return any(atom.GetSymbol() == 'C' for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = {
            3,4,11,12,13,14,19,20,21,22,23,24,25,26,27,28,29,30,
            31,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        }
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def has_isotopic_label(mol):
        # Remove explicitly isotopically labeled compounds such as deuterated SMILES ([2H]).
        return any(atom.GetIsotope() > 0 for atom in mol.GetAtoms())


    def count_heavy_atoms(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return mol.GetNumHeavyAtoms()

    def analyze_elements(df_local, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            'Total_Atom_Count': pd.Series(atom_type_counts),
            'Molecule_Count': pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df_local, smiles_column, threshold_local):
        atom_summary_local = analyze_elements(df_local, smiles_column=smiles_column)
        total_molecules = len(df_local)
        if total_molecules == 0:
            return df_local.copy(), atom_summary_local, [], 0

        rare_elements = atom_summary_local[
            atom_summary_local['Molecule_Count'] / total_molecules < threshold_local
        ].index.tolist()
        mask = []
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))

        filtered_df = df_local[mask].reset_index(drop=True)
        removed_count = len(df_local) - len(filtered_df)
        return filtered_df, atom_summary_local, rare_elements, removed_count

    stats = {'invalid': [], 'multi_component': [], 'inorganic': [], 'organometallic': [], 'ion': [], 'isotopic_label': [], 'kept': []}

    for _, row in df.iterrows():
        row_copy = row.copy()
        smi = row_copy[smiles_col]
        cano = canonicalize_smiles_inner(smi)
        if cano is None:
            stats['invalid'].append(row_copy)
            continue

        # Filter out multi-component SMILES (e.g., salts/mixtures like 'CCO.Cl')
        if '.' in str(cano):
            row_copy[smiles_col] = cano
            stats['multi_component'].append(row_copy)
            continue

        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats['invalid'].append(row_copy)
            continue

        row_copy[smiles_col] = cano

        if not is_organic(mol):
            stats['inorganic'].append(row_copy)
            continue
        if is_organometallic(mol):
            stats['organometallic'].append(row_copy)
            continue
        if is_ion(mol):
            stats['ion'].append(row_copy)
            continue
        if has_isotopic_label(mol):
            stats['isotopic_label'].append(row_copy)
            continue

        stats['kept'].append(row_copy)

    pre_dedup_df = pd.DataFrame(stats['kept'])
    pre_dedup_count = len(pre_dedup_df)
    dedup_df = pre_dedup_df.drop_duplicates(subset=[smiles_col], keep='first')
    dedup_removed = pre_dedup_count - len(dedup_df)

    cleaned_df, atom_summary, rare_elements, rare_removed = remove_rare_elements(dedup_df, smiles_col, threshold)

    cleaned_df['non_H_atom_count'] = cleaned_df[smiles_col].apply(count_heavy_atoms)
    heavy_before = len(cleaned_df)
    cleaned_df = cleaned_df[cleaned_df['non_H_atom_count'] <= max_heavy_atoms].copy()
    heavy_removed = heavy_before - len(cleaned_df)

    import inspect
    frame = inspect.currentframe().f_back
    df_name = None
    for var_name, var_val in frame.f_locals.items():
        if var_val is df:
            df_name = var_name
            break
    if df_name is None:
        df_name = 'dataset'

    if not os.path.isdir(output_path):
        os.makedirs(output_path, exist_ok=True)

    output_file = os.path.join(output_path, f'{df_name}_clean.csv')
    cleaned_df.to_csv(output_file, index=False)
    print(f'Cleaned dataset saved to {output_file}')

    print('Filter breakdown (stepwise):')
    running = len(df)

    def print_step(label, removed):
        nonlocal running
        after = running - removed
        print(f'- {label}: removed {removed} | before {running} -> after {after}')
        running = after

    print_step('Invalid SMILES', len(stats['invalid']))
    print_step('Multi-component SMILES (contains .)', len(stats['multi_component']))
    print_step('Inorganic (no carbon)', len(stats['inorganic']))
    print_step('Organometallic', len(stats['organometallic']))
    print_step('Ions (non-zero formal charge)', len(stats['ion']))
    print_step('Isotopically labeled SMILES', len(stats['isotopic_label']))
    print_step('Duplicate SMILES', dedup_removed)
    print_step(f'Rare elements (<{threshold*100:.1f}% molecule frequency)', rare_removed)
    print_step(f'Heavy atom count > {max_heavy_atoms}', heavy_removed)

    if rare_elements:
        print(f'Rare elements identified: {rare_elements}')
    else:
        print('Rare elements identified: none')

    print(f'Final retained rows: {len(cleaned_df)}')

    return cleaned_df, atom_summary

## 4. Export Curated MP Dataset


In [ ]:
union_df = combined[['SMILES', 'MP']].copy()
union_df['MP'] = pd.to_numeric(union_df['MP'], errors='coerce')
union_df = union_df.dropna(subset=['SMILES', 'MP']).copy()

os.makedirs(CURATED_DIR, exist_ok=True)
mp_df_clean, atom_summary = clean_smiles_dataset(
    union_df,
    output_path=CURATED_DIR,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
)

# Enforce exact output name used by MP workflow
curated_path = os.path.join(CURATED_DIR, CURATED_FILENAME)
mp_df_clean = mp_df_clean[['SMILES', 'MP']].copy()
mp_df_clean.to_csv(curated_path, index=False)

print(f'Saved curated MP dataset: {curated_path}')
print(f'Curated rows: {len(mp_df_clean)}')


## 5. Split for Chemprop and Export Trainable File


In [ ]:
# Match prior MP workflow: convert MP from degC to Kelvin for trainable file
df_for_split = mp_df_clean.copy()
df_for_split['MP'] = df_for_split['MP'] + 273.15

# Drop extreme target outliers above 1000 K before splitting
pre_filter_rows = len(df_for_split)
df_for_split = df_for_split[df_for_split['MP'] <= 1000].copy()
removed_rows = pre_filter_rows - len(df_for_split)
print(f'Removed rows with MP > 1000 K: {removed_rows}')

train_df, temp_df = train_test_split(df_for_split, test_size=0.2, random_state=RANDOM_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_SEED)

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

train_val_test_df = pd.concat([
    train_df[['SMILES', 'MP', 'split']],
    val_df[['SMILES', 'MP', 'split']],
    test_df[['SMILES', 'MP', 'split']]
], ignore_index=True)

os.makedirs(TRAINABLE_DIR, exist_ok=True)
trainable_path = os.path.join(TRAINABLE_DIR, TRAINABLE_FILENAME)
train_val_test_df.to_csv(trainable_path, index=False)

print(f'Saved trainable dataset: {trainable_path}')
print(f"Split sizes -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")



## 6. Final Summary


In [ ]:
print('Run summary:')
print(f'- Raw rows loaded (Original): {len(original_raw_df)}')
print(f'- Raw rows loaded (Yaws): {len(yaws_raw_df)}')
print(f'- Union rows before pKa-style clean: {len(union_df)}')
print(f'- Curated rows retained: {len(mp_df_clean)}')
print(f"- Unique SMILES in curated set: {mp_df_clean['SMILES'].nunique()}")
print(f'- Trainable rows exported: {len(train_val_test_df)}')
print(f'- Curated CSV: {os.path.join(CURATED_DIR, CURATED_FILENAME)}')
print(f'- Trainable CSV: {os.path.join(TRAINABLE_DIR, TRAINABLE_FILENAME)}')
